# Deepfake Detection — CRNN + Optuna Hyperparameter Optimization
**In-the-Wild dataset**

Author — Claudia Pletka

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pytorch-optimizer optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.2 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import io
import os
import random
import pickle
from contextlib import redirect_stdout

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.functional as AF
import torchaudio.transforms as T
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, precision_score, recall_score, f1_score
)

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from google.colab import drive

In [3]:
# ── Cell 3: Device & seed ─────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def set_seed(seed=22):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(22)

Using device: cuda
NVIDIA A100-SXM4-80GB


In [4]:
# ── Cell 4: Mount Drive & load data ───────────────────────────────────────────
drive.mount('/content/drive')

# Load In-the-Wild v2 features
TRAIN_PATH = ('/content/drive/MyDrive/ITW_Features_train_v2.pkl', 'train')
DEV_PATH   = ('/content/drive/MyDrive/ITW_Features_dev_v2.pkl',   'dev')
EVAL_PATH  = ('/content/drive/MyDrive/ITW_Features_eval_v2.pkl',  'eval')

def load_dataset(path):
    df = pd.read_pickle(path[0])
    print(f'DataFrame loaded for {path[1]} set. '
          f'Total samples: {len(df)}, shape: {df.shape}')
    return df

train_df = load_dataset(TRAIN_PATH)
dev_df   = load_dataset(DEV_PATH)
eval_df  = load_dataset(EVAL_PATH)

print('\nLabel distribution:')
print('Train:', train_df['label'].value_counts().to_dict())
print('Dev:  ', dev_df['label'].value_counts().to_dict())
print('Eval: ', eval_df['label'].value_counts().to_dict())

Mounted at /content/drive
DataFrame loaded for train set. Total samples: 26878, shape: (26878, 5)
DataFrame loaded for dev set. Total samples: 2485, shape: (2485, 5)
DataFrame loaded for eval set. Total samples: 2416, shape: (2416, 5)

Label distribution:
Train: {0: 17135, 1: 9743}
Dev:   {0: 1247, 1: 1238}
Eval:  {0: 1581, 1: 835}


In [5]:
# ── Cell 5: Prepare tensors ───────────────────────────────────────────────────
def prepare_tensors(df, feature_col='hybrid_features', label_col='label'):
    X = np.stack(df[feature_col].values).transpose(0, 2, 1).astype(np.float32)
    y = df[label_col].values
    return torch.from_numpy(X), torch.from_numpy(y)

X_train_tr, y_train_tr = prepare_tensors(train_df)
X_dev_tr,   y_dev_tr   = prepare_tensors(dev_df)
X_eval_tr,  y_eval_tr  = prepare_tensors(eval_df)

# Normalization stats (computed on train only)
epsilon  = 1e-4
x_means  = X_train_tr.mean(dim=(0, 2), keepdim=True)
x_stds   = X_train_tr.std(dim=(0, 2),  keepdim=True) + epsilon

print(f'Train : {X_train_tr.shape}, {y_train_tr.shape}')
print(f'Dev   : {X_dev_tr.shape},   {y_dev_tr.shape}')
print(f'Eval  : {X_eval_tr.shape},  {y_eval_tr.shape}')

Train : torch.Size([26878, 128, 126]), torch.Size([26878])
Dev   : torch.Size([2485, 128, 126]),   torch.Size([2485])
Eval  : torch.Size([2416, 128, 126]),  torch.Size([2416])


In [6]:
# ── Cell 6: Model components ──────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class AttentiveStatsPooling(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(input_dim, 128, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(128, input_dim, kernel_size=1),
            nn.Softmax(dim=2)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        weights = self.attention(x)
        mu = torch.sum(x * weights, dim=2)
        stdev = torch.sqrt(
            torch.sum(weights * (x ** 2), dim=2) - mu ** 2 + 1e-7
        )
        return torch.cat((mu, stdev), dim=1)

In [7]:
# ── Cell 7: CRNN model (Optuna-parameterised) ─────────────────────────────────
class Deepfake_CRNN(nn.Module):
    def __init__(self, means, stds,
                 cnn_dropout=0.2,
                 rnn_hidden_size=128,
                 rnn_num_layers=1,
                 rnn_dropout=0.0,
                 bidirectional=False):
        super().__init__()

        self.register_buffer('means', means.detach().clone().view(1, 128, 1))
        self.register_buffer('stds',  stds.detach().clone().view(1, 128, 1))

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            SEBlock(16),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(cnn_dropout),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            SEBlock(32),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(cnn_dropout),
        )

        self.bidirectional = bidirectional
        self.rnn = nn.LSTM(
            input_size=32 * 32,
            hidden_size=rnn_hidden_size,
            num_layers=rnn_num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=rnn_dropout if rnn_num_layers > 1 else 0.0,
        )

        rnn_out_dim = rnn_hidden_size * (2 if bidirectional else 1)
        self.pooling = AttentiveStatsPooling(input_dim=rnn_out_dim)

        self.fc = nn.Sequential(
            nn.Linear(rnn_out_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = (x - self.means) / (self.stds + 1e-7)
        delta1 = AF.compute_deltas(x)
        delta2 = AF.compute_deltas(delta1)
        x = torch.stack([x, delta1, delta2], dim=1)

        x = self.conv_layers(x)

        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()
        x = x.view(b, t, c * f)

        x, _ = self.rnn(x)
        x = self.pooling(x)
        return self.fc(x)

In [8]:
# ── Cell 8: Evaluation function ───────────────────────────────────────────────
def eval_model(model, dl, device, dataset_name='Dataset', show_plots=True):
    model.eval()
    all_labels, all_preds, all_scores = [], [], []

    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            probs = torch.softmax(model(xb), dim=1)
            preds = torch.argmax(probs, dim=1)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_scores.extend(probs[:, 0].cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_scores = np.array(all_scores)

    acc = accuracy_score(all_labels, all_preds)
    p   = precision_score(all_labels, all_preds, zero_division=0)
    r   = recall_score(all_labels, all_preds, zero_division=0)
    f1  = f1_score(all_labels, all_preds, zero_division=0)

    if np.any(np.isnan(all_scores)) or np.any(np.isinf(all_scores)):
        print('[!] NaN/Inf detected in scores — trial returning EER=50.0')
        return 50.0, None

    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=0)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute(fnr - fpr))] * 100

    if show_plots:
        print(f"\n{'='*10} {dataset_name} Results {'='*10}")
        print(f"EER:       {eer:.4f}%")
        print(f"Accuracy:  {acc:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"Precision: {p:.4f}")
        print(f"Recall:    {r:.4f}")
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds,
                                    target_names=['Bonafide', 'Spoof']))
        cm = confusion_matrix(all_labels, all_preds)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Bonafide', 'Spoof'],
                    yticklabels=['Bonafide', 'Spoof'])
        ax.set_title(f'Confusion Matrix: {dataset_name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.show()
        return eer, fig

    return eer, None

In [9]:
# ── Cell 9: Training loop ─────────────────────────────────────────────────────
def training_loop(epochs, model, loss_fn, opt, train_dl, dev_dl,
                  device, scheduler=None, patience=20, trial=None):
    best_dev_eer = float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    freq_mask = T.FrequencyMasking(freq_mask_param=27).to(device)
    time_mask = T.TimeMasking(time_mask_param=35).to(device)

    print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Dev EER':<10} | {'LR':<12}")
    print('-' * 52)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            xb = freq_mask(xb)
            xb = time_mask(xb)

            y_pred = model(xb)
            loss = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            running_loss += loss.item()

        if scheduler:
            scheduler.step()

        current_dev_eer, _ = eval_model(model, dev_dl, device, show_plots=False)
        avg_loss   = running_loss / len(train_dl)
        current_lr = opt.param_groups[0]['lr']

        print(f"[{epoch:02d}]   | {avg_loss:.4f}       | "
              f"{current_dev_eer:.2f}%     | {current_lr:.6f}")

        if trial is not None:
            trial.report(current_dev_eer, epoch)
            if trial.should_prune():
                print(f'[!] Trial pruned at epoch {epoch}')
                raise optuna.exceptions.TrialPruned()

        if current_dev_eer < best_dev_eer:
            best_dev_eer = current_dev_eer
            epochs_without_improvement = 0
            best_model_state = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'\n[!] Early stopping triggered at epoch {epoch}.')
                break

    if best_model_state:
        model.load_state_dict(best_model_state)
    return best_dev_eer

In [10]:
# ── Cell 10: Optuna objective ─────────────────────────────────────────────────
def make_dataloaders(batch_size):
    train_ds = TensorDataset(X_train_tr, y_train_tr)
    dev_ds   = TensorDataset(X_dev_tr,   y_dev_tr)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          pin_memory=True, num_workers=2)
    dev_dl   = DataLoader(dev_ds, batch_size=batch_size, shuffle=False,
                          pin_memory=True, num_workers=0)
    return train_dl, dev_dl

def objective(trial):
    batch_size      = trial.suggest_categorical('batch_size',      [128, 256, 512])
    lr              = trial.suggest_float('lr',                    1e-5, 1e-3,  log=True)
    weight_decay    = trial.suggest_float('weight_decay',          1e-6, 1e-3,  log=True)
    cnn_dropout     = trial.suggest_float('cnn_dropout',           0.1,  0.4)
    rnn_hidden_size = trial.suggest_categorical('rnn_hidden_size', [64, 128, 256])
    rnn_num_layers  = trial.suggest_int('rnn_num_layers',          1,    3)
    rnn_dropout     = trial.suggest_float('rnn_dropout',           0.0,  0.4)
    bidirectional   = trial.suggest_categorical('bidirectional',   [True, False])
    max_lr_factor   = trial.suggest_float('max_lr_factor',         2.0,  5.0)

    set_seed(22)
    train_dl, dev_dl = make_dataloaders(batch_size)

    model = Deepfake_CRNN(
        x_means, x_stds,
        cnn_dropout=cnn_dropout,
        rnn_hidden_size=rnn_hidden_size,
        rnn_num_layers=rnn_num_layers,
        rnn_dropout=rnn_dropout,
        bidirectional=bidirectional,
    ).to(device)

    weights = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.2)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=50,
        eta_min=1e-6
    )

    best_eer = training_loop(
        epochs=70,
        model=model,
        loss_fn=loss_fn,
        opt=opt,
        train_dl=train_dl,
        dev_dl=dev_dl,
        device=device,
        scheduler=scheduler,
        patience=15,
        trial=trial,
    )
    return best_eer

In [11]:
# ── Cell 11: Run Optuna study ─────────────
STUDY_DB   = 'sqlite:////content/drive/MyDrive/optuna_crnn_itw_v2.db'
STUDY_NAME = 'crnn_itw_v2'

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STUDY_DB,
    load_if_exists=True,
    direction='minimize',
    sampler=TPESampler(seed=22, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)

# Sanity check: warn if resuming a study with prior trials
if len(study.trials) > 0:
    print(f'Resuming existing v2 study with {len(study.trials)} prior trials.')
else:
    print('New trial')

study.optimize(
    objective,
    n_trials=50,
    timeout=3 * 3600,
    gc_after_trial=True,
)

print('\n── Optuna complete ──────────────────────────')
print(f'Best EER   : {study.best_value:.4f}%')
print(f'Best params: {study.best_params}')

[I 2026-05-07 17:18:29,841] A new study created in RDB with name: crnn_itw_v2


New trial
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6027       | 11.71%     | 0.000522
[01]   | 0.4819       | 9.21%     | 0.000521
[02]   | 0.4065       | 3.96%     | 0.000518
[03]   | 0.3915       | 4.93%     | 0.000515
[04]   | 0.3806       | 2.99%     | 0.000510
[05]   | 0.3746       | 2.99%     | 0.000505
[06]   | 0.3666       | 2.99%     | 0.000498
[07]   | 0.3710       | 3.07%     | 0.000491
[08]   | 0.3610       | 2.91%     | 0.000482
[09]   | 0.3564       | 2.67%     | 0.000473
[10]   | 0.3546       | 2.67%     | 0.000463
[11]   | 0.3645       | 2.67%     | 0.000452
[12]   | 0.3554       | 2.58%     | 0.000441
[13]   | 0.3541       | 2.67%     | 0.000428
[14]   | 0.3514       | 2.67%     | 0.000415
[15]   | 0.3500       | 2.67%     | 0.000402
[16]   | 0.3518       | 2.67%     | 0.000388
[17]   | 0.3503       | 2.67%     | 0.000373
[18]   | 0.3501       | 2.67%     | 0.000358
[19]   | 0.3482       | 2.67%  

[I 2026-05-07 17:20:31,404] Trial 0 finished with value: 2.5848142164781907 and parameters: {'batch_size': 256, 'lr': 0.0005228342106272701, 'weight_decay': 3.262005288753386e-06, 'cnn_dropout': 0.20165918818111012, 'rnn_hidden_size': 128, 'rnn_num_layers': 3, 'rnn_dropout': 0.004210749726443375, 'bidirectional': False, 'max_lr_factor': 4.235300888176507}. Best is trial 0 with value: 2.5848142164781907.


[27]   | 0.3424       | 2.67%     | 0.000213

[!] Early stopping triggered at epoch 27.
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6384       | 13.73%     | 0.000823
[01]   | 0.5133       | 10.10%     | 0.000820
[02]   | 0.4423       | 5.49%     | 0.000816
[03]   | 0.4023       | 3.88%     | 0.000811
[04]   | 0.4101       | 3.39%     | 0.000803
[05]   | 0.4001       | 3.72%     | 0.000795
[06]   | 0.3757       | 3.07%     | 0.000784
[07]   | 0.3748       | 2.99%     | 0.000773
[08]   | 0.3833       | 2.99%     | 0.000759
[09]   | 0.3688       | 2.83%     | 0.000745
[10]   | 0.3651       | 2.91%     | 0.000729
[11]   | 0.3624       | 2.83%     | 0.000712
[12]   | 0.3663       | 2.91%     | 0.000694
[13]   | 0.3589       | 2.83%     | 0.000674
[14]   | 0.3543       | 2.91%     | 0.000654
[15]   | 0.3553       | 2.75%     | 0.000633
[16]   | 0.3581       | 2.83%     | 0.000610
[17]   | 0.3565       | 2.75%     | 0.0005

[I 2026-05-07 17:22:49,941] Trial 1 finished with value: 2.6655896607431337 and parameters: {'batch_size': 512, 'lr': 0.0008235013892949465, 'weight_decay': 0.00012758912300503148, 'cnn_dropout': 0.18927348005951516, 'rnn_hidden_size': 64, 'rnn_num_layers': 2, 'rnn_dropout': 0.17102209593773626, 'bidirectional': False, 'max_lr_factor': 2.33568551565103}. Best is trial 0 with value: 2.5848142164781907.


[37]   | 0.3484       | 2.67%     | 0.000112

[!] Early stopping triggered at epoch 37.
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.5947       | 11.31%     | 0.000389
[01]   | 0.4540       | 5.82%     | 0.000388
[02]   | 0.4030       | 4.12%     | 0.000386
[03]   | 0.3905       | 3.15%     | 0.000383
[04]   | 0.3761       | 2.83%     | 0.000380
[05]   | 0.3718       | 2.91%     | 0.000376
[06]   | 0.3640       | 3.15%     | 0.000371
[07]   | 0.3608       | 2.83%     | 0.000365
[08]   | 0.3601       | 2.83%     | 0.000359
[09]   | 0.3579       | 2.83%     | 0.000352
[10]   | 0.3582       | 2.75%     | 0.000345
[11]   | 0.3543       | 2.75%     | 0.000337
[12]   | 0.3531       | 2.67%     | 0.000328
[13]   | 0.3503       | 2.75%     | 0.000319
[14]   | 0.3528       | 2.75%     | 0.000309
[15]   | 0.3510       | 2.67%     | 0.000299
[16]   | 0.3492       | 2.75%     | 0.000289
[17]   | 0.3478       | 2.67%     | 0.00027

[I 2026-05-07 17:24:39,515] Trial 2 finished with value: 2.6655896607431337 and parameters: {'batch_size': 256, 'lr': 0.00038934047188753427, 'weight_decay': 1.2224442103667365e-06, 'cnn_dropout': 0.1533277213107174, 'rnn_hidden_size': 64, 'rnn_num_layers': 1, 'rnn_dropout': 0.1937136953812435, 'bidirectional': True, 'max_lr_factor': 2.1369846277963713}. Best is trial 0 with value: 2.5848142164781907.


[27]   | 0.3450       | 2.67%     | 0.000159

[!] Early stopping triggered at epoch 27.
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.5836       | 12.12%     | 0.000234
[01]   | 0.4487       | 6.54%     | 0.000233
[02]   | 0.3985       | 5.33%     | 0.000232
[03]   | 0.3971       | 3.47%     | 0.000231
[04]   | 0.3774       | 3.47%     | 0.000229


[W 2026-05-07 17:25:08,160] Trial 3 failed with parameters: {'batch_size': 128, 'lr': 0.00023423804643141026, 'weight_decay': 0.00035283546839202354, 'cnn_dropout': 0.1375621549580086, 'rnn_hidden_size': 128, 'rnn_num_layers': 3, 'rnn_dropout': 0.24296554525679667, 'bidirectional': False, 'max_lr_factor': 4.126866241418396} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_595/701979022.py", line 45, in objective
    best_eer = training_loop(
               ^^^^^^^^^^^^^^
  File "/tmp/ipykernel_595/1294187781.py", line 31, in training_loop
    running_loss += loss.item()
                    ^^^^^^^^^^^
KeyboardInterrupt
[W 2026-05-07 17:25:08,162] Trial 3 failed with value None.


KeyboardInterrupt: 

In [12]:
# ── Cell 12: Final training with best hyperparameters ─────────────────────────
best = study.best_params
print('Training final model with:', best)

EPOCHS = 200

set_seed(22)
train_dl, dev_dl = make_dataloaders(best['batch_size'])
eval_ds  = TensorDataset(X_eval_tr, y_eval_tr)
eval_dl  = DataLoader(eval_ds, batch_size=best['batch_size'],
                      shuffle=False, pin_memory=True, num_workers=0)

model = Deepfake_CRNN(
    x_means, x_stds,
    cnn_dropout=best['cnn_dropout'],
    rnn_hidden_size=best['rnn_hidden_size'],
    rnn_num_layers=best['rnn_num_layers'],
    rnn_dropout=best['rnn_dropout'],
    bidirectional=best['bidirectional'],
).to(device)

weights = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.2)

opt = torch.optim.Adam(model.parameters(),
                       lr=best['lr'],
                       weight_decay=best['weight_decay'])

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=best['lr'] * best['max_lr_factor'],
    steps_per_epoch=1,
    epochs=EPOCHS,
    pct_start=0.2,
    div_factor=10,
    final_div_factor=100
)

print(model)

training_loop(
    epochs=EPOCHS,
    model=model,
    loss_fn=loss_fn,
    opt=opt,
    train_dl=train_dl,
    dev_dl=dev_dl,
    device=device,
    scheduler=scheduler,
    patience=20,
    trial=None,
)

Training final model with: {'batch_size': 256, 'lr': 0.0005228342106272701, 'weight_decay': 3.262005288753386e-06, 'cnn_dropout': 0.20165918818111012, 'rnn_hidden_size': 128, 'rnn_num_layers': 3, 'rnn_dropout': 0.004210749726443375, 'bidirectional': False, 'max_lr_factor': 4.235300888176507}
Deepfake_CRNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): SEBlock(
      (avg_pool): AdaptiveAvgPool2d(output_size=1)
      (fc): Sequential(
        (0): Linear(in_features=16, out_features=2, bias=False)
        (1): ReLU()
        (2): Linear(in_features=2, out_features=16, bias=False)
        (3): Sigmoid()
      )
    )
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Dropout2d(p=0.20165918818111012, inplace=False)
    (6): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), paddin

np.float64(2.10016155088853)

In [13]:
# ── Cell 13: Evaluate on Dev set ──────────────────────────────────────────────
dev_eer, dev_fig = eval_model(model, dev_dl, device, 'Development Set', show_plots=True)


========== Development Set Results ==========
EER:       2.1002%
Accuracy:  0.9795
F1 Score:  0.9793
Precision: 0.9829
Recall:    0.9758

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.98      0.98      0.98      1247
       Spoof       0.98      0.98      0.98      1238

    accuracy                           0.98      2485
   macro avg       0.98      0.98      0.98      2485
weighted avg       0.98      0.98      0.98      2485



In [14]:
# ── Cell 14: Evaluate on Eval set ─────────────────────────────────────────────
eval_eer, eval_fig = eval_model(model, eval_dl, device, 'Evaluation Set', show_plots=True)

print(f'\nGeneralization gap: {eval_eer - dev_eer:.2f}%')


========== Evaluation Set Results ==========
EER:       4.1916%
Accuracy:  0.9801
F1 Score:  0.9704
Precision: 1.0000
Recall:    0.9425

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.97      1.00      0.99      1581
       Spoof       1.00      0.94      0.97       835

    accuracy                           0.98      2416
   macro avg       0.99      0.97      0.98      2416
weighted avg       0.98      0.98      0.98      2416


Generalization gap: 2.09%


In [15]:
# ── Cell 15: Save model (v2) ─────────
torch.save(model.state_dict(), '/content/drive/MyDrive/crnn_best_itw_v2.pth')
torch.save({'means': x_means, 'stds': x_stds},
           '/content/drive/MyDrive/crnn_norm_stats_itw_v2.pth')
print('Model and normalization stats saved to Drive (v2 filenames).')

Model and normalization stats saved to Drive (v2 filenames).


In [16]:
# ── Cell 16: Reload model (v2) ────────────────────────────────────────────────
norm_stats = torch.load('/content/drive/MyDrive/crnn_norm_stats_itw_v2.pth')
model = Deepfake_CRNN(
    norm_stats['means'], norm_stats['stds'],
    **{k: best[k] for k in
       ['cnn_dropout','rnn_hidden_size','rnn_num_layers','rnn_dropout','bidirectional']}
).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/crnn_best_itw_v2.pth'))
model.eval()
print('v2 model reloaded.')

v2 model reloaded.
